In [1]:
# ============================================================
# JETRACER: ĐIỀU KHIỂN + CAMERA + DATASET RECORDER
# ============================================================

import os

# Tạm thời trong kernel, không sửa cấu hình hệ thống
os.environ.setdefault("OPENBLAS_CORETYPE", "ARMV8")

import sys
import csv
import cv2
import glob
import time
import threading
import traitlets
import ipywidgets.widgets as widgets

from pathlib import Path
from datetime import datetime


# ============================================================
# 1. NẠP MODULE JETRACER
# ============================================================
try:
    from jetracer.nvidia_racecar import NvidiaRacecar

except ModuleNotFoundError as import_error:
    if import_error.name not in (
        "jetracer",
        "jetracer.nvidia_racecar"
    ):
        raise

    candidate_files = [
        Path("/home/jetson/jetracer/jetracer/nvidia_racecar.py"),
        Path("/home/jetson/ws/jetracer/jetracer/nvidia_racecar.py"),
        Path(
            "/usr/local/lib/python3.6/dist-packages/"
            "jetracer/nvidia_racecar.py"
        ),
        Path(
            "/usr/lib/python3/dist-packages/"
            "jetracer/nvidia_racecar.py"
        ),
        Path(
            "/home/jetson/.local/lib/python3.6/"
            "site-packages/jetracer/nvidia_racecar.py"
        )
    ]

    candidate_files += [
        Path(path)
        for path in glob.glob(
            "/usr/local/lib/python3.6/dist-packages/"
            "jetracer*.egg/jetracer/nvidia_racecar.py"
        )
    ]

    jetracer_file = next(
        (
            path
            for path in candidate_files
            if path.exists()
        ),
        None
    )

    if jetracer_file is None:
        raise ModuleNotFoundError(
            "Không tìm thấy jetracer/nvidia_racecar.py"
        ) from import_error

    jetracer_root = str(
        jetracer_file.parent.parent
    )

    if jetracer_root not in sys.path:
        sys.path.insert(0, jetracer_root)

    # Xóa package sai đã được nạp trước đó
    for module_name in list(sys.modules):
        if (
            module_name == "jetracer"
            or module_name.startswith("jetracer.")
        ):
            del sys.modules[module_name]

    from jetracer.nvidia_racecar import NvidiaRacecar

    print("✅ Khôi phục JetRacer từ:")
    print(jetracer_file)


from jetcam.csi_camera import CSICamera


# ============================================================
# 2. DỌN CÁC LIÊN KẾT VÀ THREAD CŨ KHI CHẠY LẠI CELL
# ============================================================

# Dừng record cũ
try:
    old_record_event = globals().get("_record_stop_event")

    if old_record_event is not None:
        old_record_event.set()

    old_record_thread = globals().get("_record_thread")

    if (
        old_record_thread is not None
        and old_record_thread.is_alive()
    ):
        old_record_thread.join(timeout=2.0)
except Exception:
    pass


# Gỡ callback camera cũ
old_camera = globals().get("camera")

if old_camera is not None:
    for callback_name in [
        "_update_camera_preview",
        "update_camera"
    ]:
        try:
            old_callback = globals().get(callback_name)

            if old_callback is not None:
                old_camera.unobserve(
                    old_callback,
                    names="value"
                )
        except Exception:
            pass


# Gỡ liên kết tay cầm cũ
for old_link_name in [
    "steering_link",
    "throttle_link"
]:
    try:
        old_link = globals().get(old_link_name)

        if old_link is not None:
            old_link.unlink()
    except Exception:
        pass


# Dừng xe cũ trước khi tạo giao diện mới
old_car = globals().get("car")

if old_car is not None:
    try:
        old_car.throttle = 0
        old_car.steering = 0
    except Exception:
        pass


# Vô hiệu hóa các nút cũ
for old_button_name in [
    "btn_connect",
    "btn_stop",
    "btn_record_start",
    "btn_record_stop"
]:
    try:
        old_button = globals().get(old_button_name)

        if old_button is not None:
            old_button.disabled = True
    except Exception:
        pass


# ============================================================
# 3. CẤU HÌNH CHUNG
# ============================================================
INITIAL_THROTTLE_GAIN = 0.90

PREVIEW_FPS = 12
PREVIEW_JPEG_QUALITY = 55

DATASET_ROOT = Path(
    "/home/jetson/dataset_steering"
)

# 1.0 = ghi một ảnh mỗi giây
# 0.2 = ghi năm ảnh mỗi giây
RECORD_INTERVAL_SECONDS = 1.0

DATASET_JPEG_QUALITY = 85

# Góc ước tính khi car.steering = ±1
STEERING_MAX_DEG = 30.0


# ============================================================
# 4. KHỞI TẠO HOẶC TÁI SỬ DỤNG XE
# ============================================================
car_ready = (
    old_car is not None
    and hasattr(old_car, "throttle_motor")
    and hasattr(old_car, "steering_motor")
)

if car_ready:
    car = old_car
    print("✅ Tái sử dụng đối tượng xe cũ")
else:
    car = NvidiaRacecar()
    print("✅ Đã khởi tạo xe mới")

car.throttle = 0
car.steering = 0

try:
    car.steering_motor.set_pulse_width_range(
        500,
        2500
    )

    print("✅ PWM lái: 500–2500")

except Exception as error:
    print("⚠️ Không thể thiết lập PWM:", error)

car.steering_gain = -1.0
car.throttle_gain = INITIAL_THROTTLE_GAIN


# ============================================================
# 5. KHỞI TẠO TAY CẦM
# ============================================================
controller = widgets.Controller(index=0)

steering_link = None
throttle_link = None

control_output = widgets.Output()

controller_status = widgets.HTML(
    value=(
        "<b>🎮 Tay cầm:</b> "
        "Bấm một nút hoặc xoay hai cần gạt, "
        "sau đó nhấn Kích hoạt."
    )
)


# ============================================================
# 6. THANH ĐIỀU CHỈNH TỐC ĐỘ
# ============================================================
speed_slider = widgets.FloatSlider(
    value=INITIAL_THROTTLE_GAIN,
    min=0.50,
    max=1.00,
    step=0.05,
    description="Tốc độ:",
    readout=True,
    readout_format=".0%",
    continuous_update=False,
    layout=widgets.Layout(width="420px")
)


def cap_nhat_toc_do(change):
    car.throttle = 0
    car.throttle_gain = change["new"]

    with control_output:
        print(
            "⚡ Giới hạn ga: {:.0f}%".format(
                car.throttle_gain * 100
            )
        )


speed_slider.observe(
    cap_nhat_toc_do,
    names="value"
)


# ============================================================
# 7. BIẾN ĐỔI GIÁ TRỊ TAY CẦM
# ============================================================
def dieu_khien_lai(value):
    if abs(value) < 0.05:
        return 0.0

    return max(
        -1.0,
        min(1.0, value)
    )


def dieu_khien_ga(value):
    # Đảo chiều: đẩy cần lên để chạy tiến
    value = -value

    if abs(value) < 0.08:
        return 0.0

    return max(
        -1.0,
        min(1.0, value)
    )


# ============================================================
# 8. NÚT KÍCH HOẠT VÀ DỪNG KHẨN CẤP
# ============================================================
btn_connect = widgets.Button(
    description="KÍCH HOẠT ĐIỀU KHIỂN",
    button_style="success",
    layout=widgets.Layout(
        width="280px",
        height="42px"
    )
)

btn_stop = widgets.Button(
    description="DỪNG KHẨN CẤP",
    button_style="danger",
    layout=widgets.Layout(
        width="210px",
        height="42px"
    )
)


def thuc_hien_ket_noi(button):
    global steering_link
    global throttle_link

    with control_output:
        control_output.clear_output()

        if len(controller.axes) < 3:
            print(
                "❌ Chưa nhận diện đủ trục tay cầm."
            )
            print(
                "👉 Xoay cả hai cần rồi thử lại."
            )
            return

        try:
            if steering_link is not None:
                steering_link.unlink()

            if throttle_link is not None:
                throttle_link.unlink()

            car.throttle = 0
            car.steering = 0

            # Cần phải: trái/phải
            steering_link = traitlets.dlink(
                (
                    controller.axes[2],
                    "value"
                ),
                (
                    car,
                    "steering"
                ),
                transform=dieu_khien_lai
            )

            # Cần trái: tiến/lùi
            throttle_link = traitlets.dlink(
                (
                    controller.axes[1],
                    "value"
                ),
                (
                    car,
                    "throttle"
                ),
                transform=dieu_khien_ga
            )

            controller_status.value = (
                "<b style='color:green'>"
                "🎮 Tay cầm đã kết nối"
                "</b>"
            )

            print("🚀 Đã kích hoạt điều khiển")
            print("🕹️ Cần trái: Tiến/Lùi")
            print("🕹️ Cần phải: Bẻ lái")
            print(
                "⚡ Giới hạn ga: {:.0f}%".format(
                    car.throttle_gain * 100
                )
            )

        except Exception as error:
            car.throttle = 0
            print("❌ Lỗi kết nối:", error)


def dung_khan_cap(button):
    global steering_link
    global throttle_link

    try:
        if throttle_link is not None:
            throttle_link.unlink()
            throttle_link = None

        if steering_link is not None:
            steering_link.unlink()
            steering_link = None

        car.throttle = 0
        car.steering = 0

        controller_status.value = (
            "<b style='color:red'>"
            "🛑 Đã dừng xe và ngắt tay cầm"
            "</b>"
        )

        with control_output:
            control_output.clear_output()
            print("🛑 ĐÃ DỪNG KHẨN CẤP")

    except Exception as error:
        with control_output:
            print("⚠️ Lỗi dừng xe:", error)


btn_connect.on_click(thuc_hien_ket_noi)
btn_stop.on_click(dung_khan_cap)


# ============================================================
# 9. KHỞI TẠO HOẶC TÁI SỬ DỤNG CAMERA
# ============================================================
camera_ready = False

if old_camera is not None:
    try:
        camera_ready = (
            hasattr(old_camera, "cap")
            and old_camera.cap.isOpened()
        )
    except Exception:
        camera_ready = False

if camera_ready:
    camera = old_camera
    print("✅ Tái sử dụng camera cũ")
else:
    camera = CSICamera(
        width=224,
        height=224,
        capture_width=1280,
        capture_height=720,
        capture_fps=30
    )

    print("✅ Đã khởi tạo camera mới")


# ============================================================
# 10. CAMERA LIVE PREVIEW
# ============================================================
camera_view = widgets.Image(
    format="jpeg",
    width=360,
    height=360
)

camera_status = widgets.HTML(
    value="<b>📷 Camera:</b> Đang khởi động..."
)

_preview_interval = 1.0 / PREVIEW_FPS

_preview_state = {
    "last_update": 0.0
}


def _update_camera_preview(change):
    now = time.monotonic()

    if (
        now - _preview_state["last_update"]
        < _preview_interval
    ):
        return

    frame = change["new"]

    if frame is None:
        return

    success, jpeg = cv2.imencode(
        ".jpg",
        frame,
        [
            int(cv2.IMWRITE_JPEG_QUALITY),
            PREVIEW_JPEG_QUALITY
        ]
    )

    if success:
        camera_view.value = jpeg.tobytes()
        _preview_state["last_update"] = now


camera.observe(
    _update_camera_preview,
    names="value"
)

camera.running = True

camera_status.value = (
    "<b style='color:green'>"
    "📷 Camera đang hoạt động"
    "</b>"
    " — Preview khoảng {} FPS".format(
        PREVIEW_FPS
    )
)


# ============================================================
# 11. GIAO DIỆN RECORD
# ============================================================
_record_stop_event = threading.Event()
_record_thread = None

_record_state = {
    "count": 0,
    "session_dir": None
}

btn_record_start = widgets.Button(
    description="🔴 BẬT RECORD",
    button_style="success",
    layout=widgets.Layout(
        width="210px",
        height="42px"
    )
)

btn_record_stop = widgets.Button(
    description="⏹ TẮT RECORD",
    button_style="danger",
    disabled=True,
    layout=widgets.Layout(
        width="210px",
        height="42px"
    )
)

record_status = widgets.HTML(
    value="<b>⚪ Record:</b> Chưa ghi dataset"
)

record_latest = widgets.HTML(
    value="<b>Mẫu gần nhất:</b> Chưa có"
)

record_output = widgets.Output()


# ============================================================
# 12. THREAD GHI ẢNH VÀ LABELS.CSV
# ============================================================
def _record_worker(
    session_dir,
    stop_event
):
    images_dir = session_dir / "images"
    csv_path = session_dir / "labels.csv"

    start_time = time.monotonic()
    next_sample_time = start_time
    sample_index = 0

    try:
        with csv_path.open(
            "w",
            newline=""
        ) as csv_file:

            writer = csv.writer(csv_file)

            writer.writerow([
                "sample_index",
                "timestamp",
                "elapsed_seconds",
                "image_file",
                "steering_normalized",
                "steering_angle_est_deg",
                "steering_gain",
                "throttle_normalized",
                "throttle_gain"
            ])

            csv_file.flush()

            while not stop_event.is_set():
                wait_seconds = (
                    next_sample_time
                    - time.monotonic()
                )

                if wait_seconds > 0:
                    if stop_event.wait(wait_seconds):
                        break

                try:
                    frame = getattr(
                        camera,
                        "value",
                        None
                    )

                    if frame is None:
                        raise RuntimeError(
                            "Camera chưa có frame."
                        )

                    frame_copy = frame.copy()

                    steering_value = float(
                        car.steering
                    )

                    throttle_value = float(
                        car.throttle
                    )

                    steering_value = max(
                        -1.0,
                        min(1.0, steering_value)
                    )

                    steering_angle_deg = (
                        steering_value
                        * STEERING_MAX_DEG
                    )

                    elapsed_seconds = (
                        time.monotonic()
                        - start_time
                    )

                    timestamp = (
                        datetime.now().isoformat(
                            timespec="milliseconds"
                        )
                    )

                    image_name = (
                        "frame_{:06d}.jpg"
                        .format(sample_index)
                    )

                    relative_image_path = os.path.join(
                        "images",
                        image_name
                    )

                    image_path = (
                        images_dir / image_name
                    )

                    saved = cv2.imwrite(
                        str(image_path),
                        frame_copy,
                        [
                            int(
                                cv2.IMWRITE_JPEG_QUALITY
                            ),
                            DATASET_JPEG_QUALITY
                        ]
                    )

                    if not saved:
                        raise RuntimeError(
                            "Không thể lưu {}".format(
                                image_name
                            )
                        )

                    writer.writerow([
                        sample_index,
                        timestamp,
                        "{:.3f}".format(
                            elapsed_seconds
                        ),
                        relative_image_path,
                        "{:.6f}".format(
                            steering_value
                        ),
                        "{:.3f}".format(
                            steering_angle_deg
                        ),
                        "{:.3f}".format(
                            float(car.steering_gain)
                        ),
                        "{:.6f}".format(
                            throttle_value
                        ),
                        "{:.3f}".format(
                            float(car.throttle_gain)
                        )
                    ])

                    csv_file.flush()

                    sample_index += 1
                    _record_state["count"] = (
                        sample_index
                    )

                    record_status.value = (
                        "<b style='color:red'>"
                        "🔴 ĐANG RECORD"
                        "</b>"
                        " — {} mẫu".format(
                            sample_index
                        )
                    )

                    record_latest.value = (
                        "<b>Mẫu gần nhất:</b> {}"
                        " | Lái: {:.3f}"
                        " | Góc: {:.1f}°"
                        " | Ga: {:.3f}"
                        .format(
                            image_name,
                            steering_value,
                            steering_angle_deg,
                            throttle_value
                        )
                    )

                except Exception as error:
                    record_latest.value = (
                        "<b style='color:red'>"
                        "⚠️ Lỗi ghi mẫu:"
                        "</b> {}".format(error)
                    )

                next_sample_time += (
                    RECORD_INTERVAL_SECONDS
                )

                if (
                    next_sample_time
                    < time.monotonic()
                ):
                    next_sample_time = (
                        time.monotonic()
                        + RECORD_INTERVAL_SECONDS
                    )

    except Exception as error:
        record_status.value = (
            "<b style='color:red'>"
            "❌ Record bị dừng:"
            "</b> {}".format(error)
        )

        btn_record_start.disabled = False
        btn_record_stop.disabled = True


# ============================================================
# 13. BẬT/TẮT RECORD
# ============================================================
def bat_record(button):
    global _record_thread
    global _record_stop_event

    with record_output:
        record_output.clear_output()

        try:
            if (
                _record_thread is not None
                and _record_thread.is_alive()
            ):
                print("⚠️ Dataset đang được ghi.")
                return

            if not camera.running:
                camera.running = True

            if getattr(camera, "value", None) is None:
                raise RuntimeError(
                    "Camera chưa cung cấp hình ảnh."
                )

            session_name = datetime.now().strftime(
                "session_%Y%m%d_%H%M%S_%f"
            )

            session_dir = (
                DATASET_ROOT / session_name
            )

            images_dir = (
                session_dir / "images"
            )

            images_dir.mkdir(
                parents=True,
                exist_ok=False
            )

            _record_state["count"] = 0
            _record_state["session_dir"] = str(
                session_dir
            )

            _record_stop_event = (
                threading.Event()
            )

            _record_thread = threading.Thread(
                target=_record_worker,
                args=(
                    session_dir,
                    _record_stop_event
                ),
                daemon=True
            )

            _record_thread.start()

            btn_record_start.disabled = True
            btn_record_stop.disabled = False

            record_status.value = (
                "<b style='color:red'>"
                "🔴 ĐANG RECORD"
                "</b>"
            )

            record_latest.value = (
                "<b>Mẫu gần nhất:</b> "
                "Đang chờ mẫu đầu tiên..."
            )

            print("✅ Bắt đầu ghi dataset")
            print("📁", session_dir)
            print(
                "⏱️ Chu kỳ:",
                RECORD_INTERVAL_SECONDS,
                "giây/mẫu"
            )
            print(
                "📄",
                session_dir / "labels.csv"
            )

        except Exception as error:
            record_status.value = (
                "<b style='color:red'>"
                "❌ Không thể record:"
                "</b> {}".format(error)
            )

            print("❌", error)


def tat_record(button):
    global _record_thread

    _record_stop_event.set()

    if (
        _record_thread is not None
        and _record_thread.is_alive()
    ):
        _record_thread.join(timeout=2.0)

    btn_record_start.disabled = False
    btn_record_stop.disabled = True

    record_status.value = (
        "<b>⏹ ĐÃ DỪNG RECORD</b>"
        " — Tổng cộng {} mẫu".format(
            _record_state["count"]
        )
    )

    with record_output:
        print("⏹ Đã dừng record")
        print(
            "🖼️ Tổng số ảnh:",
            _record_state["count"]
        )
        print(
            "📁 Dataset:",
            _record_state["session_dir"]
        )


btn_record_start.on_click(bat_record)
btn_record_stop.on_click(tat_record)


# ============================================================
# 14. HIỂN THỊ TOÀN BỘ GIAO DIỆN
# ============================================================
camera_panel = widgets.VBox([
    widgets.HTML(
        value="<h3>📷 Camera</h3>"
    ),
    camera_view,
    camera_status
])

control_panel = widgets.VBox([
    widgets.HTML(
        value="<h3>🎮 Điều khiển xe</h3>"
    ),
    controller,
    controller_status,
    speed_slider,
    widgets.HBox([
        btn_connect,
        btn_stop
    ]),
    control_output
])

record_panel = widgets.VBox([
    widgets.HTML(
        value="<h3>💾 Dataset Recorder</h3>"
    ),
    widgets.HBox([
        btn_record_start,
        btn_record_stop
    ]),
    record_status,
    record_latest,
    record_output
])

main_interface = widgets.VBox([
    widgets.HTML(
        value=(
            "<h2>JetRacer Control Center</h2>"
            "<p>Cần trái: Tiến/Lùi — "
            "Cần phải: Bẻ lái</p>"
        )
    ),
    widgets.HBox([
        camera_panel,
        control_panel
    ]),
    record_panel
])

display(main_interface)

print("✅ Giao diện JetRacer đã sẵn sàng")
print(
    "⚡ Giới hạn ga ban đầu: {:.0f}%".format(
        car.throttle_gain * 100
    )
)
print(
    "💾 Dataset:",
    DATASET_ROOT
)

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


✅ Khôi phục JetRacer từ:
/home/jetson/jetracer/jetracer/nvidia_racecar.py
✅ Đã khởi tạo xe mới
✅ PWM lái: 500–2500
✅ Đã khởi tạo camera mới


✅ Giao diện JetRacer đã sẵn sàng
⚡ Giới hạn ga ban đầu: 90%
💾 Dataset: /home/jetson/dataset_steering
